# PLANETARY ORBIT SIMULATOR

### *A Numerical Simulation of the Sun–Earth–Moon System Using Newtonian Gravity*

**Author:** Nada Satya Maharani  
**Year:** 2026

This notebook presents an independent computational physics study
of a simplified three-body gravitational system consisting of the
Sun, Earth, and Moon.

The simulation applies Newtonian mechanics and numerical methods
to investigate orbital dynamics, velocity, acceleration, mechanical
energy, angular momentum, and conservation laws.

---

A computational physics project that models the gravitational interaction of the Sun, Earth, and Moon using Newtonian mechanics and numerical integration.

**Tools:** Python · NumPy · SciPy · Matplotlib

> **Note:** The model uses simplified initial conditions for educational and computational demonstration purposes; it is not intended to reproduce high-precision astronomical ephemerides.


# 1. Introduction

## Background

Planetary motion is a fundamental problem in classical mechanics. In this project, we simulate a simplified three-body system consisting of the **Sun, Earth, and Moon**.

Unlike a simple two-body model in which one object can be treated as fixed, this simulation calculates the gravitational interaction between **all three bodies**.

The goal is to numerically determine the position and velocity of each body over time, then analyze orbital trajectories, velocity, acceleration, mechanical energy, angular momentum, and conservation laws.

### Computational workflow

$$
\boxed{Physical Parameters→Initial Conditions→Gravity→Equations of Motion→Numerical Integration→Analysis***}
$$

The simulation is implemented in Python using NumPy, SciPy, and Matplotlib.

### Objectives

1. Implement Newtonian gravitational interaction.
2. Construct a three-body gravitational model.
3. Numerically solve the equations of motion.
4. Visualize orbital trajectories.
5. Analyze velocity and acceleration.
6. Calculate mechanical energy.
7. Calculate angular momentum.
8. Investigate conservation laws.
9. Visualize the system through animation.


In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src.simulation import create_initial_state, simulate
from src.analysis import (
    extract_solution,
    calculate_acceleration_history,
    calculate_velocity_magnitudes,
    calculate_acceleration_magnitudes,
    calculate_energy_history,
    calculate_angular_momentum_history,
    relative_conservation_error,
)
from src.visualization import (
    plot_orbits,
    plot_earth_moon_orbit,
    plot_time_series,
    plot_conservation_error,
    create_orbit_animation,
)

plt.rcParams["figure.figsize"] = (10, 7)
plt.rcParams["axes.grid"] = True

# 2. Physical Constants

Before calculating the motion of the celestial bodies, the physical parameters of the system must be defined.

The simulation uses the **International System of Units (SI)**:

- Distance → meters (m)
- Time → seconds (s)
- Mass → kilograms (kg)
- Velocity → meters per second (m/s)
- Acceleration → meters per second squared (m/s²)
- Energy → joules (J)
- Angular momentum → kg·m²/s

The gravitational constant is:

$$\boxed{G = 6.67430\times10^{-11}\;\mathrm{m^3\,kg^{-1}\,s^{-2}}}$$

Approximate masses:

$$M_S = 1.98847\times10^{30}\;\mathrm{kg}$$

$$M_E = 5.9722\times10^{24}\;\mathrm{kg}$$

$$M_M = 7.342\times10^{22}\;\mathrm{kg}$$

where $M_{S}$, $M_{E}$, and $M_{M}$ represent the masses of the Sun, Earth, and Moon.

The average Earth-Sun distance is defined using one astonomical unit:

$$
\boxed{1\ \mathrm{AU} = 1.495978707 \times 10^{11}\ \mathrm{m}}
$$

The average Earth-Moon distance is:

$$
\boxed{R_{EM} = 3.844 \times 10^{8}\ \mathrm{m}}
$$

These constants provide the physical scale required for the simulation.

### Why do we need physical constants?

The gravitational force depends directly on mass and distance. Therefore, without accurate physical parameters, the calculated gravitational acceleration would not represent the intended physical system.


In [3]:
G = 6.67430e-11

M_sun = 1.98847e30
M_earth = 5.9722e24
M_moon = 7.342e22

masses = np.array([
    M_sun,
    M_earth,
    M_moon
])

AU = 1.495978707e11
R_EM = 3.844e8

T_year = 365.25 * 24 * 3600
T_moon = 27.321661 * 24 * 3600

print(
    "Earth-Sun distance:",
    AU / 1e9,
    "million km"
)

print(
    "Earth-Moon distance:",
    R_EM / 1e6,
    "million km"
)

Earth-Sun distance: 149.5978707 million km
Earth-Moon distance: 384.4 million km


# 3. Initial Conditions

A numerical simulation requires an initial state for every body. For each object we specify its **position** and **velocity**.

The initial position determines where each body starts, while the initial velocity determines how the body begins to move.

For simplicity, the Sun is initially placed at the origin:
$$
\vec{r}_S = (0,0)
$$

The Earth is initially positioned approximately one astronomical unit from the Sun:
$$
\vec{r}_E = (1\ \mathrm{AU}, 0)
$$ 

The Moon is placed approximately one Earth–Moon distance from Earth:
$$
\vec{r}_M = \vec{r}_E + (R_{EM}, 0)
$$

The initial velocity of Earth is estimated from circular orbital motion.

For an object orbiting a central mass:
$$
\frac{GMm}{r^2} = \frac{mv^2}{r}
$$

Canceling $m$:
$$
\frac{GM}{r^2} = \frac{v^2}{r}
$$

Therefore:
$$
\boxed{v = \sqrt{\frac{GM}{r}}}
$$

For Earth:
$$
\boxed{v_E = \sqrt{\frac{GM_S}{R_{SE}}}}
$$

The Moon's velocity relative to Earth can similary be approximated by:
$$
\boxed{v_{M/E} = \sqrt{\frac{GM_E}{R_{EM}}}}
$$

Therefore, a simplified initial velocity for the Moon is:
$$
\boxed{\vec{v}_M = \vec{v}_E + \vec{v}_{M/E}}
$$

> **Assumption:** These are simplified initial conditions, not high-precision astronomical ephemeris data. The purpose is to demonstrate Newtonian orbital dynamics in a computational setting.


In [4]:
r_sun = np.array([0.0, 0.0])
r_earth = np.array([AU, 0.0])

r_moon_relative = np.array([
    R_EM,
    0.0
])

r_moon = r_earth + r_moon_relative

positions = np.array([
    r_sun,
    r_earth,
    r_moon
])

In [5]:
v_earth = np.sqrt(
    G * M_sun / AU
)

v_moon_relative = np.sqrt(
    G * M_earth / R_EM
)

velocity_sun = np.array([
    0.0,
    0.0
])

velocity_earth = np.array([
    0.0,
    v_earth
])

velocity_moon = np.array([
    0.0,
    v_earth + v_moon_relative
])

velocities = np.array([
    velocity_sun,
    velocity_earth,
    velocity_moon
])

In [6]:
print(
    "Earth orbital velocity:",
    v_earth / 1000,
    "km/s"
)

print(
    "Moon relative velocity:",
    v_moon_relative / 1000,
    "km/s"
)

Earth orbital velocity: 29.785142169221025 km/s
Moon relative velocity: 1.0183060966387332 km/s


# 4. Newtonian Gravity

Newton's law of universal gravitation gives the magnitude of the force between two masses:

$$F = G\frac{m_1m_2}{r^2}.$$

For the simulation, the vector form is required because gravitational force has both magnitude and direction:

$$\vec F_{ij}
= G\frac{m_i m_j}{|\vec r_j-\vec r_i|^3}
(\vec r_j-\vec r_i).$$

The corresponding acceleration is obtained from Newton's second law,

$$\vec F = m\vec a.$$

The acceleration function below evaluates the gravitational contribution from every other body.


In [8]:
from src.gravity import compute_accelerations

accelerations = compute_accelerations(
    positions,
    masses,
    G
)

print("Initial accelerations:")
print(accelerations)


Initial accelerations:
[[ 1.80288798e-08  0.00000000e+00]
 [-5.89709988e-03  0.00000000e+00]
 [-8.59747727e-03  0.00000000e+00]]


# 5. N-Body Differential Equations

The system contains three interacting bodies, so each body experiences gravitational acceleration from the other two.

For body $i$:

$$
\vec a_i =
G\sum_{j\ne i}
m_j
\frac{\vec r_j-\vec r_i}
{|\vec r_j-\vec r_i|^3}.
$$

This is the core of the N-body model. At every evaluation of the differential equation, the program recomputes the accelerations from the current positions.


In [9]:
from src.simulation import derivatives

initial_state = create_initial_state(
    positions,
    velocities
)

print("Number of state variables:")
print(len(initial_state))

print("State vector:")
print(initial_state)

Number of state variables:
12
State vector:
[0.00000000e+00 0.00000000e+00 1.49597871e+11 0.00000000e+00
 1.49982271e+11 0.00000000e+00 0.00000000e+00 0.00000000e+00
 0.00000000e+00 2.97851422e+04 0.00000000e+00 3.08034483e+04]


# 6. Initial State Vector

The numerical solver works with a single state vector containing all positions and velocities.

For three bodies in two dimensions there are:

$$3\times2\times2 = 12$$

state variables.

The state vector is therefore arranged as:

$$
\mathbf y =
[x_S,y_S,x_E,y_E,x_M,y_M,
v_{x,S},v_{y,S},v_{x,E},v_{y,E},v_{x,M},v_{y,M}].
$$


In [ ]:
initial_state = np.concatenate([
    positions.flatten(),
    velocities.flatten()
])

print("State vector:")
print(initial_state)


# 7. Numerical Integration

The equations of motion can be written as

$$\frac{d\vec r}{dt}=\vec v,$$

$$\frac{d\vec v}{dt}=\vec a.$$

Because the acceleration of every object depends on the changing positions of the other objects, the system is a set of coupled differential equations.

We use SciPy's `solve_ivp()` to numerically integrate the system over the selected simulation interval. The DOP853 method is used for high-order numerical integration.


In [ ]:
simulation_time = 2 * T_year

number_of_points = 6000

time = np.linspace(
    0,
    simulation_time,
    number_of_points
)

solution = solve_ivp(
    derivatives,
    (0, simulation_time),
    initial_state,
    t_eval=time,
    rtol=1e-9,
    atol=1e-9,
    method="DOP853"
)

print("Simulation finished!")
print("Number of time steps:", len(solution.t))


# 8. Extracting the Numerical Solution

The solver returns the state variables at each requested time point. These values are reshaped into separate arrays for positions and velocities.

Acceleration is then recomputed from the positions at every stored timestep so that the simulation can analyze the complete dynamical state of the system.


In [ ]:
data = solution.y.T

positions_history = data[:, :6].reshape(
    -1, 3, 2
)

velocities_history = data[:, 6:].reshape(
    -1, 3, 2
)

accelerations_history = np.array([
    compute_accelerations(p)
    for p in positions_history
])

print("Position array:",
      positions_history.shape)

print("Velocity array:",
      velocities_history.shape)

print("Acceleration array:",
      accelerations_history.shape)


# 9. Orbital Visualization

The numerical solution provides the position of each body as a function of time.

For the Earth, the trajectory is obtained by plotting:

$$x_E(t) \quad \text{against} \quad y_E(t).$$

Because the Sun–Earth distance is approximately $1.5\times10^{11}$ m while the Earth–Moon distance is only about $3.8\times10^8$ m, the Moon's orbit is difficult to see on the same scale.

Therefore, the project uses both a **global Sun–Earth visualization** and a **local Earth–Moon visualization**.


In [ ]:
plt.figure(figsize=(10, 10))

plt.scatter(
    positions_history[:, 0, 0] / AU,
    positions_history[:, 0, 1] / AU,
    s=100,
    label="Sun"
)

plt.plot(
    positions_history[:, 1, 0] / AU,
    positions_history[:, 1, 1] / AU,
    label="Earth orbit"
)

plt.plot(
    positions_history[:, 2, 0] / AU,
    positions_history[:, 2, 1] / AU,
    label="Moon orbit"
)

plt.scatter(
    positions_history[-1, 1, 0] / AU,
    positions_history[-1, 1, 1] / AU,
    s=40
)

plt.scatter(
    positions_history[-1, 2, 0] / AU,
    positions_history[-1, 2, 1] / AU,
    s=20
)

plt.xlabel("x [AU]")
plt.ylabel("y [AU]")
plt.title("Sun–Earth–Moon Orbital System")

plt.axis("equal")
plt.legend()

plt.show()


# 10. Earth–Moon Orbit

To resolve the smaller Earth–Moon system, the Moon's position is expressed relative to Earth:

$$\vec r_{M/E}=\vec r_M-\vec r_E.$$

This allows the lunar orbit to be visualized without the much larger Sun–Earth scale dominating the plot.


In [ ]:
earth_pos = positions_history[:, 1]
moon_pos = positions_history[:, 2]

moon_relative = moon_pos - earth_pos

plt.figure(figsize=(8, 8))

plt.plot(
    moon_relative[:, 0] / 1e6,
    moon_relative[:, 1] / 1e6
)

plt.scatter(
    0,
    0,
    s=100,
    label="Earth"
)

plt.xlabel("x relative to Earth [million km]")
plt.ylabel("y relative to Earth [million km]")

plt.title("Moon Orbit Around Earth")

plt.axis("equal")
plt.legend()

plt.show()


# 11. Velocity Analysis

Velocity is defined as

$$\vec v=\frac{d\vec r}{dt}.$$

In two dimensions,

$$|\vec v|=\sqrt{v_x^2+v_y^2}.$$

The simulation calculates velocity magnitude at every timestep and plots it against time. For approximately circular orbital motion, velocity is generally tangent to the trajectory.


In [ ]:
velocity_magnitude = np.linalg.norm(
    velocities_history,
    axis=2
)

time_days = solution.t / (24 * 3600)

plt.figure(figsize=(12, 6))

plt.plot(
    time_days,
    velocity_magnitude[:, 0] / 1000,
    label="Sun"
)

plt.plot(
    time_days,
    velocity_magnitude[:, 1] / 1000,
    label="Earth"
)

plt.plot(
    time_days,
    velocity_magnitude[:, 2] / 1000,
    label="Moon"
)

plt.xlabel("Time [days]")
plt.ylabel("Velocity [km/s]")

plt.title("Velocity vs Time")

plt.legend()

plt.show()


# 12. Acceleration Analysis

Acceleration is

$$\vec a=\frac{d\vec v}{dt}.$$

Its magnitude is

$$|\vec a|=\sqrt{a_x^2+a_y^2}.$$

The acceleration changes continuously because the positions of the bodies change. The Moon is influenced by both Earth and the Sun, illustrating the difference between a simple two-body model and an N-body system.


In [ ]:
acceleration_magnitude = np.linalg.norm(
    accelerations_history,
    axis=2
)

plt.figure(figsize=(12, 6))

plt.plot(
    time_days,
    acceleration_magnitude[:, 0],
    label="Sun"
)

plt.plot(
    time_days,
    acceleration_magnitude[:, 1],
    label="Earth"
)

plt.plot(
    time_days,
    acceleration_magnitude[:, 2],
    label="Moon"
)

plt.xlabel("Time [days]")
plt.ylabel("Acceleration [m/s²]")

plt.title("Acceleration vs Time")

plt.legend()

plt.show()


# 13. Energy Analysis

The total mechanical energy is

$$E=K+U.$$

The kinetic energy is

$$K=\sum_i\frac12m_iv_i^2.$$

For the three-body system, gravitational potential energy is

$$
U=-G\left(
\frac{M_SM_E}{r_{SE}}+
\frac{M_SM_M}{r_{SM}}+
\frac{M_EM_M}{r_{EM}}
\right).
$$

The simulation tracks kinetic, potential, and total energy over time.


In [ ]:
def compute_energy(positions, velocities):

    kinetic = 0.0

    for i in range(3):

        v2 = np.dot(
            velocities[i],
            velocities[i]
        )

        kinetic += 0.5 * masses[i] * v2

    potential = 0.0

    for i in range(3):

        for j in range(i + 1, 3):

            r = np.linalg.norm(
                positions[j] - positions[i]
            )

            potential -= (
                G *
                masses[i] *
                masses[j] /
                r
            )

    total = kinetic + potential

    return kinetic, potential, total


energy = np.array([
    compute_energy(p, v)
    for p, v in zip(
        positions_history,
        velocities_history
    )
])

kinetic_energy = energy[:, 0]
potential_energy = energy[:, 1]
total_energy = energy[:, 2]


# 14. Energy Conservation

For an isolated Newtonian system, total mechanical energy should remain approximately constant:

$$E(t)\approx E(0).$$

Perfect equality is not expected in a numerical simulation because numerical integration introduces small errors. The relative energy error is therefore monitored:

$$
\epsilon_E(t)=\frac{E(t)-E(0)}{|E(0)|}.
$$

A small relative error indicates that the selected numerical configuration is preserving the system's energy well.


In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    time_days,
    kinetic_energy / 1e33,
    label="Kinetic Energy"
)

plt.plot(
    time_days,
    potential_energy / 1e33,
    label="Potential Energy"
)

plt.plot(
    time_days,
    total_energy / 1e33,
    label="Total Energy",
    linewidth=2
)

plt.xlabel("Time [days]")
plt.ylabel("Energy [10³³ J]")

plt.title("Energy Conservation")

plt.legend()

plt.show()


energy_error = (
    (total_energy - total_energy[0])
    / abs(total_energy[0])
)

plt.figure(figsize=(12, 5))

plt.plot(
    time_days,
    energy_error
)

plt.xlabel("Time [days]")
plt.ylabel("Relative energy error")

plt.title("Numerical Energy Conservation")

plt.show()

print(
    "Maximum relative energy error:",
    np.max(np.abs(energy_error))
)


# 15. Angular Momentum

Angular momentum is

$$\vec L=\vec r\times\vec p,$$

with

$$\vec p=m\vec v.$$

For this two-dimensional simulation, the relevant component is

$$L_z=m(xv_y-yv_x).$$

The total angular momentum is the sum of the angular momentum contributions from the three bodies.


In [ ]:
def compute_angular_momentum(
    positions,
    velocities
):

    L_total = 0.0

    for i in range(3):

        x, y = positions[i]
        vx, vy = velocities[i]

        Lz = masses[i] * (
            x * vy -
            y * vx
        )

        L_total += Lz

    return L_total


angular_momentum = np.array([
    compute_angular_momentum(p, v)
    for p, v in zip(
        positions_history,
        velocities_history
    )
])

angular_momentum_error = (
    (angular_momentum - angular_momentum[0])
    / abs(angular_momentum[0])
)


# 16. Angular Momentum Conservation

Gravity is a central force, so the torque associated with the mutual gravitational interaction is zero for the isolated system. Consequently, total angular momentum should remain approximately constant.

The relative error is monitored using

$$
\epsilon_L(t)=\frac{L(t)-L(0)}{|L(0)|}.
$$

Conservation checks are important because a visually plausible orbit is not by itself proof that a numerical simulation is physically reliable.


In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    time_days,
    angular_momentum / 1e40
)

plt.xlabel("Time [days]")
plt.ylabel(
    "Angular Momentum [10⁴⁰ kg m²/s]"
)

plt.title(
    "Total Angular Momentum"
)

plt.show()


# 17. Velocity & Acceleration Vectors

The vector visualization shows the direction of Earth's velocity and acceleration along its orbit.

For approximately circular motion, velocity is approximately tangent to the trajectory, while gravitational acceleration points toward the attracting mass. The vectors provide an intuitive visual check of the dynamics represented by the equations.


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    time_days,
    angular_momentum_error
)

plt.xlabel("Time [days]")
plt.ylabel("Relative error")

plt.title(
    "Angular Momentum Conservation"
)

plt.show()

print(
    "Maximum relative angular momentum error:",
    np.max(
        np.abs(angular_momentum_error)
    )
)


indices = np.linspace(
    0,
    len(time) - 1,
    40,
    dtype=int
)

plt.figure(figsize=(10, 10))

earth_x = positions_history[
    indices, 1, 0
] / AU

earth_y = positions_history[
    indices, 1, 1
] / AU

earth_vx = velocities_history[
    indices, 1, 0
] / 3e4

earth_vy = velocities_history[
    indices, 1, 1
] / 3e4

earth_ax = accelerations_history[
    indices, 1, 0
] / 0.006

earth_ay = accelerations_history[
    indices, 1, 1
] / 0.006

plt.plot(
    positions_history[:, 1, 0] / AU,
    positions_history[:, 1, 1] / AU,
    label="Earth orbit"
)

plt.quiver(
    earth_x,
    earth_y,
    earth_vx,
    earth_vy,
    label="Velocity"
)

plt.quiver(
    earth_x,
    earth_y,
    earth_ax,
    earth_ay,
    label="Acceleration"
)

plt.scatter(
    0,
    0,
    s=100,
    label="Sun"
)

plt.xlabel("x [AU]")
plt.ylabel("y [AU]")

plt.title(
    "Earth Orbit with Velocity and Acceleration"
)

plt.axis("equal")
plt.legend()

plt.show()


# 18. Animated Orbit Simulator

The numerical integration already contains the position of each body at every stored timestep. The animation does not perform a new physics calculation; it simply displays those previously calculated positions sequentially.

**Simulation ≠ Animation**

The simulation calculates the physics, while the animation visualizes the numerical solution.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)

ax.set_xlabel("x [AU]")
ax.set_ylabel("y [AU]")

ax.set_title("Sun–Earth–Moon Newtonian Gravity Simulation")

ax.set_aspect("equal")

ax.plot(
    positions_history[:, 1, 0] / AU,
    positions_history[:, 1, 1] / AU,
    alpha=0.3,
    label="Earth trajectory"
)

sun_dot, = ax.plot(
    [],
    [],
    "o",
    markersize=12,
    label="Sun"
)

earth_dot, = ax.plot(
    [],
    [],
    "o",
    markersize=6,
    label="Earth"
)

moon_dot, = ax.plot(
    [],
    [],
    "o",
    markersize=3,
    label="Moon"
)

earth_trail, = ax.plot([], [], linewidth=1)
moon_trail, = ax.plot([], [], linewidth=1)

ax.legend()


def init():

    sun_dot.set_data([], [])
    earth_dot.set_data([], [])
    moon_dot.set_data([], [])

    earth_trail.set_data([], [])
    moon_trail.set_data([], [])

    return (
        sun_dot,
        earth_dot,
        moon_dot,
        earth_trail,
        moon_trail
    )


def update(frame):

    sun = positions_history[frame, 0] / AU
    earth = positions_history[frame, 1] / AU
    moon = positions_history[frame, 2] / AU

    sun_dot.set_data(
        [sun[0]],
        [sun[1]]
    )

    earth_dot.set_data(
        [earth[0]],
        [earth[1]]
    )

    moon_dot.set_data(
        [moon[0]],
        [moon[1]]
    )

    earth_trail.set_data(
        positions_history[
            :frame, 1, 0
        ] / AU,

        positions_history[
            :frame, 1, 1
        ] / AU
    )

    moon_trail.set_data(
        positions_history[
            :frame, 2, 0
        ] / AU,

        positions_history[
            :frame, 2, 1
        ] / AU
    )

    return (
        sun_dot,
        earth_dot,
        moon_dot,
        earth_trail,
        moon_trail
    )


animation = FuncAnimation(
    fig,
    update,
    frames=range(
        0,
        len(time),
        10
    ),
    init_func=init,
    interval=20,
    blit=True
)

HTML(
    animation.to_jshtml()
)


# 19. Physics Summary & Conclusion

The project follows a complete computational-physics workflow:

**Physical Constants → Initial Conditions → Newtonian Gravity → N-Body Equations → Numerical Integration → Orbital Data → Velocity & Acceleration → Energy → Angular Momentum → Conservation Laws → Animation**

The project demonstrates how classical mechanics can be translated into a numerical model and analyzed with scientific Python tools.

### Limitations

- Initial conditions are simplified rather than taken from a precision ephemeris.
- The model is two-dimensional.
- Relativistic effects are ignored.
- Bodies are treated as point masses.
- Numerical accuracy depends on the chosen solver tolerances and timestep sampling.

These limitations are intentional for an educational computational model and can be addressed in future extensions.


In [ ]:
print("=" * 60)
print("PLANETARY ORBIT SIMULATOR")
print("Sun + Earth + Moon")
print("=" * 60)

print()

print("Simulation duration :",
      simulation_time / T_year,
      "years")

print()

print("Earth initial velocity :",
      np.linalg.norm(velocities[1]) / 1000,
      "km/s")

print("Moon initial velocity :",
      np.linalg.norm(velocities[2]) / 1000,
      "km/s")

print()

print("Initial total energy :",
      total_energy[0],
      "J")

print("Final total energy   :",
      total_energy[-1],
      "J")

print()

print("Energy conservation error :",
      np.max(np.abs(energy_error)))

print(
    "Angular momentum error :",
    np.max(
        np.abs(
            angular_momentum_error
        )
    )
)

print("=" * 60)
